### MaxCut Experiment (Direct Probability Model)
This section shows a MaxCut experiment using a simple DirectProbModel to solve multi-way MaxCut on the Graph_bat dataset, with configurable group counts 
k


In [1]:
import sys
from pathlib import Path
current = Path.cwd()
root_path = current.parents[3]  
sys.path.append(str(root_path))
import torch
import torch.nn as nn
from src.maxcut import loss_maxcut_onehot_qubo, Net, DirectProbModel
from src import from_file_to_graph, init, get_device, run_qubo, Layer, LayerType, Datasets

/home/guohao/miniconda3/envs/kgrouping/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialization and experiment setup:

In [2]:
init(cuda_index=1, reproducibility=False)
device = get_device()  
data_path = Datasets.Graph_bat.path
data = from_file_to_graph(data_path, True, True).to(get_device())
num_group = 3                                                                          #k=2、3、4、5、6
num_nodes = data.num_v
net = DirectProbModel(num_nodes, num_group).to(device)
x = torch.randn(1, 1).to(device)
loss, outs, eval_result = run_qubo(
    "maxcut",
    net,
    x,
    data,
    15000,
    0.95,
    "rmsprop",
    True,
    loss_maxcut_onehot_qubo,
)


[INFO] Using CUDA device: NVIDIA GeForce RTX 3090 (Index: 1)


INFO: 71 loops found


 15%|█▍        | 2225/15000 [00:02<00:11, 1145.03it/s]

Epoch: 2000.00 | obj Loss: -1651.17 | gini Loss: 8.90 | 


 28%|██▊       | 4215/15000 [00:04<00:09, 1136.76it/s]

...

Epoch: 12000.00 | obj Loss: -1653.35 | gini Loss: 7.50 | 


 98%|█████████▊| 14627/15000 [00:07<00:00, 3760.06it/s]

Epoch: 14000.00 | obj Loss: -1651.55 | gini Loss: 8.10 | 


100%|██████████| 15000/15000 [00:07<00:00, 1940.66it/s]


+------------[MaxCut Evaluation]------------+
Cut Edges: 829/1003 (82.7%)
Group Distribution: 56 vs 24
Unconverged Nodes: 5
+--------------------------------------------+


### MaxCut Experiment (GNN-Based Model)
This section shows a MaxCut experiment using a GNN-based Net model (with GraphSAGE layers) to solve 2-way MaxCut on the same dataset, using a more complex architecture.

In [ ]:
data_path = Datasets.Graph_bat.path
graph = from_file_to_graph(data_path, True, True).to(get_device())
init_feature_dim = 2048
num_group = 2
x = torch.rand((graph.num_v, init_feature_dim)).to(get_device())
layers = [
    Layer(LayerType.GRAPHSAGE, init_feature_dim, 256, hidden_channels=512, num_layers=2, jk="last", drop_rate=0, act="leaky_relu"),
    Layer(LayerType.LINEAR, 256, num_group, use_bn=True, drop_rate=0.2),
    ]
net = Net(layers).to(get_device())

gini_cof_lambda = lambda e, n: (-200 + 0.25 * e) / 10

loss, outs, eval_result = run_qubo(
    "maxcut",
    net,
    x,
    graph,
    2400,
    2.5e-5,
    "AdamW",
    True,
    loss_maxcut_onehot_qubo,
    )

INFO: 71 loops found


  0%|          | 0/2400 [00:00<?, ?it/s]

 85%|████████▍ | 2033/2400 [00:08<00:01, 239.30it/s]

Epoch: 2000.00 | obj Loss: -1299.58 | gini Loss: 4.06 | 


100%|██████████| 2400/2400 [00:09<00:00, 247.20it/s]


+------------[MaxCut Evaluation]------------+
Cut Edges: 650/1003 (64.8%)
Group Distribution: 88 vs 43
Unconverged Nodes: 7
+--------------------------------------------+
